In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Reduces TF logging to errors only

In [4]:

import cv2
import pickle
import os
import numpy as np

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Reduces TF logging to errors only

with open('image_classification/cnn_model.pkl', 'rb') as file:
    model=pickle.load(file)

img1='image_processing/Skin cancer ISIC The International Skin Imaging Collaboration/images/ISIC_0000001.jpg'
img0='image_processing/healthy_skin/3fdg6hg.jpg'
img_can=cv2.imread(img1)
img_hel=cv2.imread(img0)

if img_can is not None:
    resize_can=cv2.resize(img_can, (28, 28), interpolation=cv2.INTER_AREA)
    gray=cv2.cvtColor(resize_can, cv2.COLOR_BGR2GRAY)
    normalized=gray.astype('float32')/255.0
    val=np.array(normalized)
    val=val.reshape(-1, 28, 28)
    prop=model.predict(val)
    if prop > 0.5:
        print('having skin cancer')
    else:
        print('not having skin cancer')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
having skin cancer


In [ ]:
#1. Reversing the Normalization
import cv2
import numpy as np
import matplotlib.pyplot as plt

def denormalize_image(img_tensor):
    # 1. Remove the batch dimension [1, 150, 150, 3] -> [150, 150, 3]
    img = np.squeeze(img_tensor)
    
    # 2. Reverse the /255.0 scaling# 1. Get the prediction
img_processed = preprocess_image("skin_test.jpg") # your normalization step
prob = model.predict(img_processed)

# 2. Reverse the image for display
display_img = denormalize_image(img_processed)

# 3. Only draw the box if the probability is high (Cancer Detected)
if prob[0][0] > 0.5:
    print(f"Cancer Detected: {prob[0][0]*100:.2f}%")
    
    # Logic to get heatmap (requires a Grad-CAM function)
    # For now, let's assume we use the display_img
    result_img = draw_red_rectangle(display_img, generated_heatmap)
    
    
    # Show the result
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.title("Detection Result")
    plt.show()
else:
    print("Result: Healthy Skin")
    plt.imshow(cv2.cvtColor(display_img, cv2.COLOR_BGR2RGB))
    plt.show()
    img = (img * 255).astype(np.uint8)
    
    # 3. Convert RGB (TensorFlow default) to BGR (OpenCV default)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    return img

#2. Drawing a Box using Grad-CAM (The "Where")
def draw_red_rectangle(original_img, heatmap):
    # 1. Threshold the heatmap to find the "hottest" area
    # This creates a binary mask of the suspicious region
    _, thresh = cv2.threshold(heatmap, 155, 255, cv2.THRESH_BINARY)
    
    # 2. Find the contours (shapes) of that area
    contours, _ = cv2.find_centers(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        # 3. Find the largest contour (the main cancer site)
        largest_cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_cnt)
        
        # 4. Draw a RED rectangle (0, 0, 255) in BGR
        cv2.rectangle(original_img, (x, y), (x + w, y + h), (0, 0, 255), 3)
        cv2.putText(original_img, 'MALIGNANT AREA', (x, y-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    
    return original_img

#3. Complete Prediction & Visualization Loop


In [ ]:
import pickle